# Autoencoders: code companion

This notebook accompanies `lecture_notes/19_autoencoders.pdf` and turns the main concepts into small, executable experiments.

The central idea is always the same: train a network to reconstruct its own input. Since the training target is `x` itself, we do not need labels to fit the model. Dataset labels appear here only for visualization and interpretation of the latent space.

We will use scikit-learn's `digits` dataset (8 x 8 images) to avoid downloads and keep classroom execution time short.


## 1. Setup

Each image is a vector with 64 pixels normalized to the interval `[0, 1]`. This corresponds to the role of `x in R^d` in the notes, with `d = 64`.


In [ ]:
import os
import tempfile
import warnings

os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "matplotlib"))

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.manifold import TSNE
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (8, 4)


In [ ]:
digits = load_digits()
X = digits.data.astype("float32") / 16.0
Y = digits.target
images = digits.images.astype("float32") / 16.0

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.25, random_state=RANDOM_STATE, stratify=Y
)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("pixel min/max:", X_train.min(), X_train.max())


In [ ]:
def plot_digit_grid(vectors, titles=None, n=10, image_shape=(8, 8), cmap="gray"):
    n = min(n, len(vectors))
    fig, axes = plt.subplots(1, n, figsize=(1.2 * n, 1.4))
    if n == 1:
        axes = [axes]
    for i, ax in enumerate(axes):
        ax.imshow(vectors[i].reshape(image_shape), cmap=cmap, vmin=0, vmax=1)
        if titles is not None:
            ax.set_title(str(titles[i]), fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    return fig

plot_digit_grid(X_train[:12], titles=y_train[:12], n=12)
plt.suptitle("Samples from the digits dataset", y=1.05)
plt.show()


## 2. Architecture: encoder, bottleneck, and decoder

An autoencoder combines two learned functions:

- `z = f_phi(x)`: the encoder compresses the input into a latent code.
- `x_hat = g_psi(z)`: the decoder tries to reconstruct the input from that code.

In scikit-learn we do not explicitly define `Encoder` and `Decoder` modules, but the architecture `64 -> 32 -> latent_dim -> 32 -> 64` has exactly this interpretation. The middle layer is the bottleneck.


In [ ]:
def make_autoencoder(latent_dim=8, hidden_dim=64, max_iter=250, random_state=RANDOM_STATE):
    return MLPRegressor(
        hidden_layer_sizes=(hidden_dim, latent_dim, hidden_dim),
        activation="relu",
        solver="adam",
        learning_rate_init=1e-3,
        alpha=1e-4,
        batch_size=128,
        max_iter=max_iter,
        early_stopping=False,
        validation_fraction=0.15,
        n_iter_no_change=25,
        random_state=random_state,
        verbose=False,
    )

ae = make_autoencoder(latent_dim=8, hidden_dim=64, max_iter=250)

# Unsupervised training: the input is also the target.
ae.fit(X_train, X_train)

X_hat = np.clip(ae.predict(X_test), 0, 1)
print("epochs run:", ae.n_iter_)
print("test reconstruction MSE:", mean_squared_error(X_test, X_hat))


## 3. Reconstruction loss

In the notes, the objective function is a loss between `x` and `x_hat`. Here we use MSE:

$$L(x, \hat{x}) = \frac{1}{d}\lVert x - \hat{x}\rVert_2^2.$$

Notice that digit labels do not enter the optimization. The model only learns to preserve enough information to reconstruct the image.


In [ ]:
def reconstruction_mse(X_true, X_recon):
    return np.mean((X_true - X_recon) ** 2, axis=1)

errors = reconstruction_mse(X_test, X_hat)
print("mean error per image:", errors.mean())
print("lowest error:", errors.min())
print("highest error:", errors.max())


In [ ]:
n = 10
fig, axes = plt.subplots(2, n, figsize=(1.2 * n, 2.8))
for i in range(n):
    axes[0, i].imshow(X_test[i].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    axes[0, i].axis("off")
    axes[1, i].imshow(X_hat[i].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("original", fontsize=10)
axes[1, 0].set_ylabel("recon", fontsize=10)
plt.suptitle("Original vs. reconstruction")
plt.tight_layout()
plt.show()


## 4. The effect of the bottleneck

The latent dimension `k` controls compression. Small values force compact codes, but may lose details; large values reduce reconstruction error, but may approach an almost identical copy of the input.


In [ ]:
latent_dims = [2, 4, 8, 16, 32]
results = []
models = {}

for k in latent_dims:
    model = make_autoencoder(latent_dim=k, hidden_dim=64, max_iter=250, random_state=RANDOM_STATE + k)
    model.fit(X_train, X_train)
    recon = np.clip(model.predict(X_test), 0, 1)
    mse = mean_squared_error(X_test, recon)
    results.append((k, mse, model.n_iter_))
    models[k] = model

for k, mse, n_iter in results:
    print(f"latent_dim={k:2d} | test MSE={mse:.5f} | iter={n_iter}")

plt.plot([r[0] for r in results], [r[1] for r in results], marker="o")
plt.xlabel("latent dimension k")
plt.ylabel("reconstruction MSE")
plt.title("Trade-off between compression and reconstruction")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
example_idx = 3
fig, axes = plt.subplots(1, len(latent_dims) + 1, figsize=(1.5 * (len(latent_dims) + 1), 1.8))
axes[0].imshow(X_test[example_idx].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
axes[0].set_title("original")
axes[0].axis("off")

for ax, k in zip(axes[1:], latent_dims):
    recon = np.clip(models[k].predict(X_test[example_idx:example_idx + 1]), 0, 1)
    ax.imshow(recon.reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"k={k}")
    ax.axis("off")

plt.suptitle("Same image reconstructed with different bottlenecks")
plt.tight_layout()
plt.show()


## 5. Latent space

To visualize what the encoder learned, we need to extract the activations of the bottleneck layer. The function below manually runs the first layers of the `MLPRegressor` until it reaches the latent code.


In [ ]:
def relu(a):
    return np.maximum(a, 0)

def encode_with_sklearn_mlp(model, X):
    # Return the bottleneck-layer activation for a 64 -> h -> k -> h -> 64 architecture.
    h1 = relu(X @ model.coefs_[0] + model.intercepts_[0])
    z = relu(h1 @ model.coefs_[1] + model.intercepts_[1])
    return z

best_ae = models[8]
Z_test = encode_with_sklearn_mlp(best_ae, X_test)
print("latent space shape:", Z_test.shape)


In [ ]:
# Since k=8, we project the codes to 2D with t-SNE only for visualization.
tsne = TSNE(n_components=2, init="pca", learning_rate="auto", perplexity=30, random_state=RANDOM_STATE)
Z_2d = tsne.fit_transform(Z_test)

plt.figure(figsize=(7, 5))
scatter = plt.scatter(Z_2d[:, 0], Z_2d[:, 1], c=y_test, cmap="tab10", s=18, alpha=0.8)
plt.legend(*scatter.legend_elements(), title="digit", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.title("t-SNE of autoencoder latent codes")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.tight_layout()
plt.show()


Labels were used only to color the plot. If points from the same digit appear close together, this suggests that the autoencoder learned representations related to the visual structure of digits, even without supervision.

Important: t-SNE is a visualization tool. Do not treat global distances in the plot as precise metrics of the original latent space.


## 6. Denoising autoencoder

In a denoising autoencoder, we corrupt the input before the encoder, but compare the output with the clean image:

$$\tilde{x} = x + \epsilon, \quad \hat{x} = g_\psi(f_\phi(\tilde{x})), \quad L = L(x, \hat{x}).$$

This detail is crucial: the loss target is `x`, not `x_tilde`.


In [ ]:
def add_gaussian_noise(X, noise_factor=0.35, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    noisy = X + rng.normal(loc=0.0, scale=noise_factor, size=X.shape)
    return np.clip(noisy, 0, 1)

noise_factor = 0.35
X_train_noisy = add_gaussian_noise(X_train, noise_factor=noise_factor, rng=rng)
X_test_noisy = add_gaussian_noise(X_test, noise_factor=noise_factor, rng=rng)

dae = make_autoencoder(latent_dim=8, hidden_dim=64, max_iter=250, random_state=123)

# Corrupted input, clean target.
dae.fit(X_train_noisy, X_train)
X_denoised = np.clip(dae.predict(X_test_noisy), 0, 1)

print("MSE of noisy image vs. clean image: ", mean_squared_error(X_test, X_test_noisy))
print("MSE of reconstruction vs. clean image:", mean_squared_error(X_test, X_denoised))


In [ ]:
n = 10
fig, axes = plt.subplots(3, n, figsize=(1.2 * n, 4.0))
for i in range(n):
    axes[0, i].imshow(X_test[i].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    axes[0, i].axis("off")
    axes[1, i].imshow(X_test_noisy[i].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    axes[1, i].axis("off")
    axes[2, i].imshow(X_denoised[i].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    axes[2, i].axis("off")
axes[0, 0].set_ylabel("clean", fontsize=10)
axes[1, 0].set_ylabel("noisy", fontsize=10)
axes[2, 0].set_ylabel("recon", fontsize=10)
plt.suptitle("Denoising: noisy input, clean target")
plt.tight_layout()
plt.show()


## 7. Reconstruction error as an anomaly signal

If we train an autoencoder only on data considered normal, it tends to reconstruct samples similar to that set more accurately. Inputs outside the training distribution often have higher reconstruction error.

Here, as a didactic demonstration, we train only on digits `0` through `4` and treat `5` through `9` as out of distribution.


In [ ]:
normal_mask = Y <= 4
X_normal = X[normal_mask]
y_normal = Y[normal_mask]
X_anom = X[~normal_mask]
y_anom = Y[~normal_mask]

Xn_train, Xn_test, yn_train, yn_test = train_test_split(
    X_normal, y_normal, test_size=0.30, random_state=RANDOM_STATE, stratify=y_normal
)

anom_ae = make_autoencoder(latent_dim=8, hidden_dim=64, max_iter=250, random_state=7)
anom_ae.fit(Xn_train, Xn_train)

normal_recon = np.clip(anom_ae.predict(Xn_test), 0, 1)
anom_recon = np.clip(anom_ae.predict(X_anom), 0, 1)

normal_err = reconstruction_mse(Xn_test, normal_recon)
anom_err = reconstruction_mse(X_anom, anom_recon)

print("mean error on seen digits (0-4):", normal_err.mean())
print("mean error on unseen digits (5-9):", anom_err.mean())


In [ ]:
plt.hist(normal_err, bins=30, alpha=0.7, label="seen: 0-4", density=True)
plt.hist(anom_err, bins=30, alpha=0.7, label="unseen: 5-9", density=True)
plt.xlabel("MSE per image")
plt.ylabel("density")
plt.title("Distribution of reconstruction error")
plt.legend()
plt.show()


## 8. Common pitfalls to check in code

- Loss and output activation must be compatible. For MSE, a linear output is acceptable; for BCE, we usually use a sigmoid output.
- Very low reconstruction error does not guarantee a good representation; the bottleneck may be too large.
- In denoising, the loss must compare the reconstruction with the clean input.
- Blurry or imperfect reconstructions are expected when the bottleneck is small or the model is simple.
- The latent space of a standard autoencoder can be irregular; generating random samples from the latent space is not as reliable as in a VAE.


## 9. Connecting with the PyTorch implementation from the notes

The same structure would appear in PyTorch as:

```python
class AutoEncoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 28 * 28),
            nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28)),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)
```

In the vanilla training loop, the loss compares `recon` with `imgs`. In the denoising loop, the encoder receives `imgs_noisy`, but the loss compares `recon` with `imgs`.


## Quick exercises

1. Change `latent_dim=8` to `latent_dim=2` and observe the impact on reconstructions and on t-SNE.
2. Increase `noise_factor` in the denoising autoencoder. At what point does the task become too difficult?
3. Use the latent codes `Z_test` as input to a simple classifier. Is the accuracy close to the one obtained with the original pixels?
4. In the anomaly section, change which digits are considered normal and observe how the error distributions change.
